# Three-tier Qwen quality-preserving ModernBERT router

This is notebook v3. It replaces the narrow 7.0B-versus-8.2B candidate
panel in notebook 02 with three separated Qwen2.5 tiers:

| Tier | Candidate | Exact parameters | Offline precision |
|---|---|---:|---:|
| small | `Qwen2.5-1.5B` | 1.54B | NF4 4-bit |
| middle | `Qwen2.5-3B` | 3.09B | NF4 4-bit |
| strong | `Qwen2.5-7B` | 7.61B | NF4 4-bit |

Qwen does not publish literal 1B, 3B, and 8B Qwen2.5 checkpoints. These
are the closest official instruction-tuned tiers. Revisions are pinned.

Notebook 02's latest seed-44 run was **not successful**: it routed 22.21%
of 2,805 sealed-test prompts, gained 43 answers, lost 52, and missed the
macro-quality, routed-precision, and guarded-dataset gates. Its 42.55 ms
measured router p50 was below the 52.14 ms break-even point, but its
67.23 ms p95 was above break-even. That is evidence of a real safety and
tail-latency problem, not an investor-ready win.

V3 separates two phases:

1. **Offline evidence collection:** run each pinned Qwen candidate on the
   same 900 prompts and cache outcomes in Google Drive.
2. **Router evaluation:** use only prompt text and prompt-token count to
   predict fallback-relative safety. Candidate latency remains analytical.

A numerical example: if the 7.61B fallback is correct and the 1.54B model
is wrong, the small model's safety label is 0. If both are correct, its
label is 1. Since $1.54/7.61\approx20\%$, the analytical latency gap is
materially larger than the old $7.0/8.2\approx85\%$ ratio.

> The default 900-prompt panel is a Colab pilot. It can reject a bad router,
> but one passing run is not investment-grade evidence. Repeat all seeds,
> report dataset-OOD separately, and expand the evidence panel before a
> production claim.


## 1. Set up Colab

Use a GPU runtime. The candidate models are loaded one at a time and then
released, so the 7.61B checkpoint fits a typical Colab T4 in 4-bit. The
cloned repository remains the source of the router, calibration, gates,
analytical latency equations, and artifact export logic.


In [ ]:
%cd /content
!test -d /content/LLM_Router || git clone --branch develop https://github.com/BrunoVitti96/LLM-router.git /content/LLM_Router
!git -C /content/LLM_Router pull --ff-only origin develop
%cd /content/LLM_Router
%pip install -q -U ".[notebook,qwen-evaluation]"

import importlib
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/LLM_Router")
SOURCE_ROOT = str(PROJECT_ROOT / "src")
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)
for module_name in list(sys.modules):
    if module_name == "llm_router" or module_name.startswith("llm_router."):
        del sys.modules[module_name]
importlib.invalidate_caches()
llm_router = importlib.import_module("llm_router")
print(f"Router package ready from {Path(llm_router.__file__).resolve()}")


## 2. Freeze the run and evidence contracts

Change only `RUN_ID` between router experiments. Candidate outcomes are
independent of the router split seed, so a completed Qwen evidence cache
is reused. `random` asks prompt-level feasibility; `dataset_ood` holds out
entire task families and is a harder, separate claim.

The prompt pool contains 150 examples from each of six tasks (900 total).
Three candidates therefore produce $900	imes3=2{,}700$ scored outcomes.


In [ ]:
import gc
import hashlib
import json
import re
import shutil
from dataclasses import replace
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from IPython.display import display
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from llm_router.config import DEFAULT_CONFIG
from llm_router.experiment_comparison import (
    build_setup_comparison,
    choose_validation_setup,
    combine_threshold_searches,
)
from llm_router.hybrid_inference import (
    HybridModernBERTRouterRuntime,
    create_gradio_demo,
)
from llm_router.modernbert_poc import (
    export_modernbert_hybrid_poc,
    train_modernbert_hybrid_poc,
)
from llm_router.oracle import oracle_choices
from llm_router.public_benchmark import (
    EconomicsScenario,
    ModelProfile,
    export_public_benchmark,
    make_complete_panel,
    run_public_benchmark,
    select_validation_policy,
    simulate_economics,
    split_benchmark,
)
from llm_router.router_overhead import benchmark_modernbert_overhead
from llm_router.utils.training import seed_everything

RUN_SPECS = {
    "qwen25_random_seed_42": ("random", 42),
    "qwen25_random_seed_43": ("random", 43),
    "qwen25_random_seed_44": ("random", 44),
    "qwen25_dataset_ood_seed_42": ("dataset_ood", 42),
    "qwen25_dataset_ood_seed_43": ("dataset_ood", 43),
    "qwen25_dataset_ood_seed_44": ("dataset_ood", 44),
}
RUN_ID = "qwen25_random_seed_42"
SPLIT_MODE, SEED = RUN_SPECS[RUN_ID]

SAMPLES_PER_DATASET = 150
COLLECTION_SEED = 20260821
GENERATION_BATCH_SIZE = 4
MAX_INPUT_TOKENS = 2048
EPOCHS = 8
MINIMUM_EPOCHS = 2
EARLY_STOPPING_PATIENCE = 2
MEASURE_ROUTER_OVERHEAD = True
LAUNCH_INTERACTIVE_DEMO = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "Choose Runtime > Change runtime type > GPU."

SETUP_SPECS = {
    "hybrid_r4": {
        "lora_r": 4,
        "lora_alpha": 8,
        "oracle_auxiliary_weight": 0.25,
        "dataset_balanced_sampling": False,
    },
    "safety_only_r4": {
        "lora_r": 4,
        "lora_alpha": 8,
        "oracle_auxiliary_weight": 0.0,
        "dataset_balanced_sampling": False,
    },
    "hybrid_r8": {
        "lora_r": 8,
        "lora_alpha": 16,
        "oracle_auxiliary_weight": 0.25,
        "dataset_balanced_sampling": False,
    },
}

CANDIDATES = {
    "Qwen2.5-1.5B": {
        "repo": "Qwen/Qwen2.5-1.5B-Instruct",
        "revision": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
        "parameters_billions": 1.54,
    },
    "Qwen2.5-3B": {
        "repo": "Qwen/Qwen2.5-3B-Instruct",
        "revision": "aa8e72537993ba99e69dfaafa59ed015b17504d1",
        "parameters_billions": 3.09,
    },
    "Qwen2.5-7B": {
        "repo": "Qwen/Qwen2.5-7B-Instruct",
        "revision": "a09a35458c702b33eeacc393d103063234e8bc28",
        "parameters_billions": 7.61,
    },
}
SELECTED_MODELS = tuple(CANDIDATES)

DATASET_REVISIONS = {
    "gsm8k": "740312add88f781978c0658806c59bc2815b9866",
    "arc_challenge": "210d026faf9955653af8916fad021475a3f00453",
    "mmlu": "c30699e8356da336a370243923dbaf21066bb9fe",
    "boolq": "35b264d03638db9f4ce671b711558bf7ff0f80d5",
    "hellaswag": "218ec52e09a7e7462a5400043bb9a69a41d06b76",
    "winogrande": "01e74176c63542e6b0bcb004dcdea22d94fb67b5",
}
PROMPT_TEMPLATE_ID = "qwen25-tier-pilot-json-free-v1-2026-08-21"

evidence_contract = {
    "candidates": CANDIDATES,
    "dataset_revisions": DATASET_REVISIONS,
    "samples_per_dataset": SAMPLES_PER_DATASET,
    "collection_seed": COLLECTION_SEED,
    "prompt_template_id": PROMPT_TEMPLATE_ID,
    "generation": {
        "do_sample": False,
        "batch_size": GENERATION_BATCH_SIZE,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "quantization": "bitsandbytes NF4 4-bit",
    },
}
EVIDENCE_TAG = hashlib.sha256(
    json.dumps(evidence_contract, sort_keys=True).encode("utf-8")
).hexdigest()[:16]

try:
    from google.colab import drive

    drive.mount("/content/drive")
    EVIDENCE_ROOT = Path("/content/drive/MyDrive/llm_router_qwen_tiers")
except ImportError:
    EVIDENCE_ROOT = PROJECT_ROOT / "qwen_tier_cache"
EVIDENCE_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = PROJECT_ROOT / "reports_benchmark" / RUN_ID

def log_stage(stage, **values):
    timestamp = datetime.now(timezone.utc).strftime("%H:%M:%S UTC")
    details = " | ".join(f"{key}={value}" for key, value in values.items())
    print(f"[{timestamp}] {stage}" + (f" | {details}" if details else ""))

seed_everything(SEED)
log_stage(
    "contracts frozen",
    run_id=RUN_ID,
    split=SPLIT_MODE,
    evidence_tag=EVIDENCE_TAG,
    gpu=torch.cuda.get_device_name(0),
)
display(pd.DataFrame(CANDIDATES).T)


## 3. Build the pinned six-domain prompt pool

The same sampled source indices are used for every candidate. Multiple
choice prompts require a final letter; GSM8K requires `FINAL: <number>`.
Scoring is deterministic and independent of answer length or latency.
Source dataset revisions are part of `EVIDENCE_TAG`, so changing a dataset,
sample count, model revision, prompt, or generation policy creates a new
cache rather than silently reusing incompatible evidence.


In [ ]:
LETTERS = tuple("ABCDEFGHIJKLMNOPQRSTUVWXYZ")

def sampled_indices(length, dataset_offset):
    rng = np.random.default_rng(COLLECTION_SEED + dataset_offset)
    count = min(SAMPLES_PER_DATASET, length)
    return np.sort(rng.choice(length, size=count, replace=False))

def mc_prompt(stem, choices):
    rendered = "\n".join(
        f"{LETTERS[index]}. {choice}" for index, choice in enumerate(choices)
    )
    return (
        f"{stem.strip()}\n\n{rendered}\n\n"
        "Choose the single best option. End with exactly FINAL: <letter>."
    )

pool_rows = []

gsm = load_dataset(
    "openai/gsm8k", "main", split="test", revision=DATASET_REVISIONS["gsm8k"]
)
for index in sampled_indices(len(gsm), 1):
    row = gsm[int(index)]
    match = re.search(r"####\s*([^\n]+)", row["answer"])
    assert match, "GSM8K answer did not contain its documented final marker."
    pool_rows.append(
        {
            "example_id": f"gsm8k::test::{index}",
            "dataset": "gsm8k",
            "source_split": "test",
            "source_index": int(index),
            "prompt": (
                f"Solve this grade-school math problem carefully:\n{row['question']}\n\n"
                "End with exactly FINAL: <number>."
            ),
            "target": match.group(1).strip(),
            "target_type": "number",
        }
    )

arc = load_dataset(
    "allenai/ai2_arc",
    "ARC-Challenge",
    split="test",
    revision=DATASET_REVISIONS["arc_challenge"],
)
for index in sampled_indices(len(arc), 2):
    row = arc[int(index)]
    labels = list(row["choices"]["label"])
    choices = list(row["choices"]["text"])
    answer_position = labels.index(row["answerKey"])
    pool_rows.append(
        {
            "example_id": f"arc_challenge::test::{index}",
            "dataset": "arc_challenge",
            "source_split": "test",
            "source_index": int(index),
            "prompt": mc_prompt(row["question"], choices),
            "target": LETTERS[answer_position],
            "target_type": "choice",
        }
    )

mmlu = load_dataset(
    "cais/mmlu", "all", split="test", revision=DATASET_REVISIONS["mmlu"]
)
for index in sampled_indices(len(mmlu), 3):
    row = mmlu[int(index)]
    stem = f"Subject: {row['subject']}\n\n{row['question']}"
    pool_rows.append(
        {
            "example_id": f"mmlu::test::{index}",
            "dataset": "mmlu",
            "source_split": "test",
            "source_index": int(index),
            "prompt": mc_prompt(stem, list(row["choices"])),
            "target": LETTERS[int(row["answer"])],
            "target_type": "choice",
        }
    )

boolq = load_dataset(
    "google/boolq", split="validation", revision=DATASET_REVISIONS["boolq"]
)
for index in sampled_indices(len(boolq), 4):
    row = boolq[int(index)]
    stem = f"Passage: {row['passage']}\n\nQuestion: {row['question']}"
    pool_rows.append(
        {
            "example_id": f"boolq::validation::{index}",
            "dataset": "boolq",
            "source_split": "validation",
            "source_index": int(index),
            "prompt": mc_prompt(stem, ["No", "Yes"]),
            "target": "B" if bool(row["answer"]) else "A",
            "target_type": "choice",
        }
    )

hellaswag = load_dataset(
    "Rowan/hellaswag",
    split="validation",
    revision=DATASET_REVISIONS["hellaswag"],
)
for index in sampled_indices(len(hellaswag), 5):
    row = hellaswag[int(index)]
    stem = f"Complete the scenario:\n{row['ctx']}"
    pool_rows.append(
        {
            "example_id": f"hellaswag::validation::{index}",
            "dataset": "hellaswag",
            "source_split": "validation",
            "source_index": int(index),
            "prompt": mc_prompt(stem, list(row["endings"])),
            "target": LETTERS[int(row["label"])],
            "target_type": "choice",
        }
    )

winogrande = load_dataset(
    "allenai/winogrande",
    "winogrande_xl",
    split="validation",
    revision=DATASET_REVISIONS["winogrande"],
)
for index in sampled_indices(len(winogrande), 6):
    row = winogrande[int(index)]
    stem = f"Choose the option that correctly fills the blank:\n{row['sentence']}"
    pool_rows.append(
        {
            "example_id": f"winogrande::validation::{index}",
            "dataset": "winogrande",
            "source_split": "validation",
            "source_index": int(index),
            "prompt": mc_prompt(stem, [row["option1"], row["option2"]]),
            "target": "A" if str(row["answer"]) == "1" else "B",
            "target_type": "choice",
        }
    )

prompt_pool = pd.DataFrame(pool_rows).sort_values("example_id").reset_index(drop=True)
expected_prompts = SAMPLES_PER_DATASET * len(DATASET_REVISIONS)
assert len(prompt_pool) == expected_prompts
assert prompt_pool.example_id.is_unique
assert prompt_pool.groupby("dataset").size().eq(SAMPLES_PER_DATASET).all()
prompt_pool.to_parquet(EVIDENCE_ROOT / f"prompt_pool__{EVIDENCE_TAG}.parquet")
display(prompt_pool.groupby(["dataset", "target_type"]).size().rename("prompts"))


## 4. Collect or resume the three candidate outcome panels

Each model runs deterministically in NF4 4-bit, one model at a time. A
checkpoint is written after every dataset, so a disconnected Colab can
resume. These generations create **quality labels**, not latency labels.
The later router still uses only model facts and explicit hardware
assumptions for candidate latency.

The scorer first looks for the required `FINAL:` marker. A malformed
response scores 0. For example, target `B` with output `FINAL: B` scores
1; output `I think B, but FINAL: A` scores 0.


In [ ]:
NUMBER_PATTERN = r"[-+]?(?:\d[\d,]*\.?\d*|\.\d+)(?:[eE][-+]?\d+)?"

def canonical_number(value):
    cleaned = str(value).strip().replace(",", "")
    try:
        number = Decimal(cleaned)
    except InvalidOperation:
        return None
    return number.normalize()

def extract_number(text):
    final = re.findall(rf"FINAL\s*:\s*({NUMBER_PATTERN})", str(text), re.I)
    candidates = final or re.findall(NUMBER_PATTERN, str(text))
    return canonical_number(candidates[-1]) if candidates else None

def extract_choice(text):
    value = str(text).upper()
    final = re.findall(r"FINAL\s*:\s*\(?([A-Z])\)?", value)
    if final:
        return final[-1]
    answer = re.findall(r"(?:ANSWER|OPTION)\s*(?:IS|:)\s*\(?([A-Z])\)?", value)
    if answer:
        return answer[-1]
    standalone = re.findall(r"\b([A-Z])\b", value)
    return standalone[-1] if standalone else None

def score_prediction(target_type, target, prediction):
    if target_type == "number":
        predicted = extract_number(prediction)
        expected = canonical_number(target)
    else:
        predicted = extract_choice(prediction)
        expected = str(target).upper()
    return float(predicted is not None and predicted == expected)

def collect_candidate(model_name, spec):
    cache_path = EVIDENCE_ROOT / f"{model_name.replace('.', '_')}__{EVIDENCE_TAG}.parquet"
    if cache_path.exists():
        collected = pd.read_parquet(cache_path)
        assert collected.evidence_tag.eq(EVIDENCE_TAG).all()
        assert collected.model_revision.eq(spec["revision"]).all()
    else:
        collected = pd.DataFrame()

    completed_ids = set(collected.example_id) if not collected.empty else set()
    missing = prompt_pool.loc[~prompt_pool.example_id.isin(completed_ids)]
    if missing.empty:
        log_stage("candidate cache complete", model=model_name, rows=len(collected))
        return collected

    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )
    tokenizer = AutoTokenizer.from_pretrained(
        spec["repo"], revision=spec["revision"], use_fast=True
    )
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        spec["repo"],
        revision=spec["revision"],
        quantization_config=quantization,
        torch_dtype=compute_dtype,
        device_map="auto",
    ).eval()

    log_stage("candidate collection started", model=model_name, missing=len(missing))
    for dataset_name, task_rows in missing.groupby("dataset", sort=True):
        generated_rows = []
        max_new_tokens = 192 if dataset_name == "gsm8k" else 32
        task_rows = task_rows.reset_index(drop=True)
        for start in range(0, len(task_rows), GENERATION_BATCH_SIZE):
            batch = task_rows.iloc[start : start + GENERATION_BATCH_SIZE]
            chat_prompts = [
                tokenizer.apply_chat_template(
                    [
                        {
                            "role": "system",
                            "content": "Follow the requested final-answer format exactly.",
                        },
                        {"role": "user", "content": prompt},
                    ],
                    tokenize=False,
                    add_generation_prompt=True,
                )
                for prompt in batch.prompt
            ]
            encoded = tokenizer(
                chat_prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_INPUT_TOKENS,
            ).to(model.device)
            input_width = encoded.input_ids.shape[1]
            with torch.inference_mode():
                output_ids = model.generate(
                    **encoded,
                    do_sample=False,
                    max_new_tokens=max_new_tokens,
                    use_cache=True,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            completion_ids = output_ids[:, input_width:]
            predictions = tokenizer.batch_decode(
                completion_ids, skip_special_tokens=True
            )
            prompt_lengths = encoded.attention_mask.sum(dim=1).cpu().numpy()
            for position, (_, row) in enumerate(batch.iterrows()):
                prediction = predictions[position].strip()
                generated_rows.append(
                    {
                        **row.to_dict(),
                        "model": model_name,
                        "model_repo": spec["repo"],
                        "model_revision": spec["revision"],
                        "prediction": prediction,
                        "score": score_prediction(
                            row.target_type, row.target, prediction
                        ),
                        "prompt_tokens": int(prompt_lengths[position]),
                        "completion_tokens": len(
                            tokenizer.encode(
                                prediction, add_special_tokens=False
                            )
                        ),
                        "recorded_cost": 0.0,
                        "evidence_tag": EVIDENCE_TAG,
                    }
                )
        collected = pd.concat(
            [collected, pd.DataFrame(generated_rows)], ignore_index=True
        ).drop_duplicates(["example_id", "model"], keep="last")
        collected.to_parquet(cache_path, index=False)
        log_stage(
            "candidate dataset checkpoint",
            model=model_name,
            dataset=dataset_name,
            completed=len(collected),
        )

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return collected

candidate_frames = [
    collect_candidate(model_name, spec)
    for model_name, spec in CANDIDATES.items()
]
records = pd.concat(candidate_frames, ignore_index=True)
expected_rows = len(prompt_pool) * len(CANDIDATES)
assert len(records) == expected_rows
assert records.groupby("example_id").model.nunique().eq(len(CANDIDATES)).all()
display(
    records.pivot_table(
        index="dataset", columns="model", values="score", aggfunc="mean"
    ).style.format("{:.1%}")
)


## 5. Audit the evidence and attach analytical latency

The panel rejects missing prompt/model pairs and duplicate content crossing
splits. Candidate latency is computed from parameter count, 4-bit weight
precision, prompt size, expected output length, and dated hardware
assumptions. Realized completion length is excluded.

For active parameters $P$, effective compute $F$, bandwidth $B$, and
precision $b$, the core terms are:

$$t_{compute/token}=\frac{2P}{F},\qquad
t_{memory/pass}=\frac{Pb/8}{B}.$$

The code multiplies every recorded completion length by 100 and requires
analytical latency to remain identical. That is the leakage check.


In [ ]:
analytical_records = records.assign(
    formatted_prompt=records.prompt,
    ground_truth=records.target,
    source_file=f"drive-cache::{EVIDENCE_TAG}",
)
profiles = tuple(
    ModelProfile(
        name=name,
        input_price_per_million=0.0,
        output_price_per_million=0.0,
        parameters_billions=spec["parameters_billions"],
        active_parameters_billions=spec["parameters_billions"],
        architecture="autoregressive",
        weight_bits=4,
    )
    for name, spec in CANDIDATES.items()
)
scenario = EconomicsScenario(
    name="qwen25-three-tier-analytical-latency-poc",
    as_of="2026-08-21",
    profiles=profiles,
    latency_method="analytical",
    effective_tflops=60.0,
    memory_bandwidth_gbps=900.0,
    fixed_model_overhead_s=0.015,
    router_overhead_s=0.004,
    output_base_tokens=24.0,
    output_tokens_per_prompt_token=0.20,
    output_min_tokens=16,
    output_max_tokens=256,
    notes="Pinned Qwen quality evidence; analytical candidate latency only.",
)
simulated = simulate_economics(analytical_records, scenario)
panel = make_complete_panel(simulated, models=SELECTED_MODELS)

counterfactual = analytical_records.copy()
counterfactual["completion_tokens"] *= 100
counterfactual_simulated = simulate_economics(counterfactual, scenario)
assert np.allclose(
    simulated.simulated_latency_s,
    counterfactual_simulated.simulated_latency_s,
), "Leakage check failed: completion length changed analytical latency."

split = split_benchmark(panel, mode=SPLIT_MODE, seed=SEED)
split_indices = {"train": split.train, "validation": split.validation, "test": split.test}
prompt_hashes = {
    name: set(panel.examples.iloc[indices].prompt_hash)
    for name, indices in split_indices.items()
}
assert prompt_hashes["train"].isdisjoint(prompt_hashes["validation"])
assert prompt_hashes["train"].isdisjoint(prompt_hashes["test"])
assert prompt_hashes["validation"].isdisjoint(prompt_hashes["test"])
if SPLIT_MODE == "dataset_ood":
    assert set(split.train_datasets).isdisjoint(split.validation_datasets)
    assert set(split.train_datasets).isdisjoint(split.test_datasets)

quality_summary = records.groupby("model").score.mean().rename("quality")
latency_summary = simulated.groupby("model").simulated_latency_s.mean().rename(
    "mean_analytical_latency_s"
)
panel_summary = pd.concat([quality_summary, latency_summary], axis=1)
panel_summary["latency_vs_7b"] = (
    panel_summary.mean_analytical_latency_s
    / panel_summary.loc["Qwen2.5-7B", "mean_analytical_latency_s"]
)
display(panel_summary.style.format("{:.2%}", subset=["quality", "latency_vs_7b"]))
display(
    pd.DataFrame(
        {
            "split": list(split_indices),
            "prompts": [len(indices) for indices in split_indices.values()],
        }
    )
)
log_stage(
    "Leakage check and split audit passed",
    complete_prompts=len(panel.examples),
    train=len(split.train),
    validation=len(split.validation),
    test=len(split.test),
)


## 6. Verify validation-only oracle headroom and scenario sensitivity

The hindsight oracle can see recorded outcomes and chooses the fastest
candidate that preserves the training-selected fallback's quality. It is
training-only. If it cannot save latency, no learned prompt-only router can
rescue the panel under this objective.

Example: if 1.5B and 7B both score 1, the oracle chooses 1.5B. If only 7B
scores 1, it chooses 7B. V3 requires positive validation oracle savings in
every declared analytical scenario before spending time on ModernBERT.


In [ ]:
def validation_oracle_metrics(candidate_panel):
    fallback_index = int(candidate_panel.score[split.train].mean(axis=0).argmax())
    choices = oracle_choices(
        candidate_panel.score,
        candidate_panel.latency,
        fallback_index=fallback_index,
    )
    indices = split.validation
    rows_index = np.arange(len(indices))
    selected = choices[indices]
    fallback_quality = candidate_panel.score[indices, fallback_index].mean()
    oracle_quality = candidate_panel.score[indices][rows_index, selected].mean()
    fallback_latency = candidate_panel.latency[indices, fallback_index].mean()
    oracle_latency = candidate_panel.latency[indices][rows_index, selected].mean()
    return {
        "fallback_model": candidate_panel.models[fallback_index],
        "quality_retention": oracle_quality / max(fallback_quality, 1e-12),
        "latency_savings": 1 - oracle_latency / fallback_latency,
        "fallback_usage": np.mean(selected == fallback_index),
    }

sensitivity_settings = {
    "balanced": {},
    "compute_conservative": {"effective_tflops": 40.0},
    "bandwidth_conservative": {"memory_bandwidth_gbps": 600.0},
    "larger_fixed_overhead": {"fixed_model_overhead_s": 0.050},
    "longer_outputs": {
        "output_base_tokens": 48.0,
        "output_tokens_per_prompt_token": 0.35,
    },
}
sensitivity_rows = []
for scenario_name, overrides in sensitivity_settings.items():
    variant = replace(scenario, name=scenario_name, **overrides)
    variant_records = simulate_economics(analytical_records, variant)
    variant_panel = make_complete_panel(variant_records, models=SELECTED_MODELS)
    sensitivity_rows.append(
        {"scenario": scenario_name, **validation_oracle_metrics(variant_panel)}
    )
sensitivity = pd.DataFrame(sensitivity_rows)
assert (sensitivity.latency_savings > 0).all(), (
    "No robust oracle headroom. Stop: this panel cannot support the routing claim."
)
display(sensitivity.style.format({"quality_retention": "{:.2%}", "latency_savings": "{:.2%}"}))


## 7. Understand the objective, loss, and calibration

ModernBERT predicts one replacement-safety probability for each
non-fallback candidate:

$$y_m(x)=\mathbf 1[Q_m(x)\ge Q_f(x)-\epsilon_q].$$

With default $\epsilon_q=0$, matching the fallback is safe. Independent,
class-balanced binary cross-entropy trains the deployed safety heads. A
training-only hindsight-oracle head adds 0.25 times its quality-first
latency-regret loss in the hybrid setups:

$$\mathcal L_{train}=\mathcal L_{safety}+0.25\mathcal L_{oracle}.$$

Platt scalers are fitted per candidate. Threshold search uses out-of-fold
validation probabilities, so a row never calibrates its own confidence.
The sealed test remains unopened until one setup and a contiguous block of
at least two feasible thresholds are frozen.

Numerical example: if the small candidate is safe on 30 of 100 training
prompts, its positive BCE weight is $70/30=2.33$. A safe example predicted
at 0.8 contributes $2.33[-\log(0.8)]\approx0.52$ before averaging.


## 8. Train every declared router setup

The three candidate tiers are fixed evidence. The validation comparison
asks whether the oracle auxiliary loss helps and whether rank-8 LoRA adds
useful router capacity. Test outcomes do not select among these setups.


In [ ]:
def make_epoch_logger(setup_name):
    def report(row):
        marker = "BEST" if row["is_best_epoch"] else "    "
        stop = " | early-stop" if row["will_stop_early"] else ""
        print(
            f"[{setup_name}] epoch={int(row['epoch'])}/{EPOCHS} {marker} "
            f"train={row['train_total_loss']:.4f} "
            f"validation={row['validation_total_loss']:.4f} "
            f"best_epoch={int(row['best_epoch_so_far'])} "
            f"seconds={row['epoch_seconds']:.1f} "
            f"examples_per_second={row['train_examples_per_second']:.1f}"
            f"{stop}"
        )
    return report

trainings = {}
setup_configs = {}
for setup_name, spec in SETUP_SPECS.items():
    setup_config = replace(
        DEFAULT_CONFIG,
        seed=SEED,
        lora_r=spec["lora_r"],
        lora_alpha=spec["lora_alpha"],
    )
    setup_configs[setup_name] = setup_config
    log_stage("router training started", setup=setup_name, **spec)
    training = train_modernbert_hybrid_poc(
        panel,
        split,
        config=setup_config,
        epochs=EPOCHS,
        batch_size=8,
        learning_rate=1e-4,
        head_learning_rate=2e-4,
        minimum_epochs=MINIMUM_EPOCHS,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        quality_epsilon=0.0,
        safety_loss_weight=1.0,
        oracle_auxiliary_weight=spec["oracle_auxiliary_weight"],
        dataset_balanced_sampling=spec["dataset_balanced_sampling"],
        device=DEVICE,
        progress_callback=make_epoch_logger(setup_name),
    )
    training.model.to("cpu")
    torch.cuda.empty_cache()
    trainings[setup_name] = training
    log_stage(
        "router training finished",
        setup=setup_name,
        best_epoch=training.best_epoch,
        epochs=training.epochs_completed,
        truncation=f"{training.input_diagnostics['truncation_rate']:.2%}",
    )


## 9. Compare setups on validation and freeze one policy

Activation requires aggregate and macro quality bounds, harm-rate control,
routed safety precision, the guarded-dataset floor when enough examples
exist, positive savings at both 4 ms and 20 ms overhead, and at least two
adjacent passing thresholds. An isolated lucky threshold fails closed.


In [ ]:
policy_kwargs = dict(
    objective="latency",
    minimum_quality_retention=DEFAULT_CONFIG.minimum_quality_retention,
    confidence=DEFAULT_CONFIG.quality_confidence,
    validation_quality_margin=DEFAULT_CONFIG.validation_quality_margin,
    minimum_predicted_savings=DEFAULT_CONFIG.minimum_predicted_speedup,
    router_overhead_s=scenario.router_overhead_s,
    conservative_router_overhead_s=DEFAULT_CONFIG.conservative_router_overhead_s,
    minimum_macro_quality_retention=DEFAULT_CONFIG.minimum_macro_quality_retention,
    maximum_quality_loss_rate_ucl=DEFAULT_CONFIG.maximum_quality_loss_rate_ucl,
    minimum_routed_safety_precision_lcb=DEFAULT_CONFIG.minimum_routed_safety_precision_lcb,
    minimum_guarded_dataset_quality_retention_lcb=(
        DEFAULT_CONFIG.minimum_guarded_dataset_quality_retention_lcb
    ),
    minimum_guarded_dataset_prompts=DEFAULT_CONFIG.minimum_guarded_dataset_prompts,
    minimum_consecutive_feasible_thresholds=(
        DEFAULT_CONFIG.minimum_consecutive_feasible_thresholds
    ),
    seed=SEED,
)
selections = {}
for setup_name, training in trainings.items():
    selection = select_validation_policy(
        panel,
        split,
        routing_probabilities=training.safety_probabilities,
        router_name=f"modernbert_qwen_tier_router__{setup_name}",
        **policy_kwargs,
    )
    selections[setup_name] = selection
    log_stage(
        "validation policy evaluated",
        setup=setup_name,
        active=selection.router_active,
        threshold=selection.selected_threshold,
        reason="; ".join(selection.failure_reasons) or "all gates passed",
    )

setup_comparison = build_setup_comparison(trainings, selections)
setup_threshold_search = combine_threshold_searches(selections)
display(
    setup_comparison.sort_values(
        ["router_active", "conservative_resource_savings"], ascending=False
    ).style.format(
        {
            "quality_retention_lcb": "{:.2%}",
            "routed_safety_precision_lcb": "{:.2%}",
            "safe_opportunity_recall": "{:.2%}",
            "routed_fraction": "{:.2%}",
            "nominal_resource_savings": "{:.2%}",
            "conservative_resource_savings": "{:.2%}",
        }
    )
)

SELECTED_SETUP = choose_validation_setup(setup_comparison)
setup_comparison["selected_for_test"] = setup_comparison.setup.eq(SELECTED_SETUP)
selected_training = trainings[SELECTED_SETUP]
selected_config = setup_configs[SELECTED_SETUP]
selected_spec = SETUP_SPECS[SELECTED_SETUP]
frozen_validation_policy = selections[SELECTED_SETUP]
log_stage(
    "POLICY FROZEN — sealed test may now open",
    setup=SELECTED_SETUP,
    active=frozen_validation_policy.router_active,
    threshold=frozen_validation_policy.selected_threshold,
    feasible_block=int(
        setup_comparison.set_index("setup").loc[
            SELECTED_SETUP, "feasible_block_size"
        ]
    ),
)


## 10. Measure ModernBERT overhead only

This measures tokenization, host-to-device transfer, and ModernBERT
inference on the named Colab GPU. It does not time any Qwen candidate and
cannot change the already-frozen threshold. The measured p50 and p95 are
compared with the eventual policy-specific break-even overhead.


In [ ]:
router_overhead_benchmark = None
if MEASURE_ROUTER_OVERHEAD:
    validation_examples = panel.examples.iloc[split.validation]
    router_overhead_benchmark = benchmark_modernbert_overhead(
        selected_training.model,
        selected_training.tokenizer,
        validation_examples.prompt.to_numpy(),
        validation_examples.prompt_tokens.to_numpy(),
        device=DEVICE,
        max_input_tokens=selected_config.max_input_tokens,
        timed_requests=min(100, len(validation_examples)),
        warmup_requests=10,
        seed=SEED,
    )
    measured = router_overhead_benchmark.summary["end_to_end_ms"]
    log_stage(
        "ModernBERT overhead measured",
        p50_ms=f"{measured['p50']:.2f}",
        p95_ms=f"{measured['p95']:.2f}",
        candidate_latency="analytical-only",
        policy_changed=False,
    )
    display(pd.DataFrame(router_overhead_benchmark.summary).loc[
        ["mean", "p50", "p95", "maximum"],
        ["end_to_end_ms", "model_only_ms"],
    ])


## 11. Open the sealed test exactly once

A non-trivial router passes only when every predeclared sealed-test safety
and savings gate passes. Fallback-only behavior is a safe operational
result, but it is not evidence that learned routing works.


In [ ]:
result = run_public_benchmark(
    panel,
    split,
    routing_probabilities=selected_training.safety_probabilities,
    router_name="modernbert_qwen_tier_router",
    selected_setup=SELECTED_SETUP,
    decision_metadata={
        "router_input_tokens": selected_training.router_input_lengths,
        "router_was_truncated": selected_training.router_was_truncated,
    },
    **policy_kwargs,
)
assert result.selected_threshold == frozen_validation_policy.selected_threshold
assert result.router_active == frozen_validation_policy.router_active

router_metrics = result.summary.loc["modernbert_qwen_tier_router"]
routed = result.decisions.selected_model.ne(result.fallback_model)
gained = result.decisions.quality_delta.gt(0)
lost = result.decisions.quality_delta.lt(0)
log_stage(
    "sealed test complete",
    single_run_passed=result.single_run_passed,
    routed=f"{routed.mean():.2%}",
    gained=int(gained.sum()),
    lost=int(lost.sum()),
    net_answers=int(gained.sum() - lost.sum()),
    retention_lcb=f"{router_metrics.quality_retention_lcb:.2%}",
    savings_4ms=f"{router_metrics.resource_savings:.2%}",
    savings_20ms=f"{router_metrics.conservative_resource_savings:.2%}",
)
display(result.summary)
display(result.router_overhead_sensitivity)
if result.failure_reasons:
    print("Failure reasons:")
    for reason in result.failure_reasons:
        print("-", reason)

break_even_ms = float(
    result.router_overhead_sensitivity.break_even_router_overhead_ms.iloc[0]
)
overhead_rows = [
    {"source": "frozen nominal assumption", "overhead_ms": 4.0},
    {"source": "frozen conservative assumption", "overhead_ms": 20.0},
]
if router_overhead_benchmark is not None:
    measured = router_overhead_benchmark.summary["end_to_end_ms"]
    overhead_rows.extend(
        [
            {"source": "measured ModernBERT p50", "overhead_ms": measured["p50"]},
            {"source": "measured ModernBERT p95", "overhead_ms": measured["p95"]},
        ]
    )
overhead_comparison = pd.DataFrame(overhead_rows)
overhead_comparison["break_even_ms"] = break_even_ms
overhead_comparison["below_break_even"] = (
    overhead_comparison.overhead_ms < break_even_ms
)
display(overhead_comparison)


## 12. Diagnose task mix, tier usage, and truncation

Investor-facing averages must not hide that one task supplies all gains or
another absorbs most losses. The tables below expose net answers and
conservative savings by dataset, selected-tier counts, and truncation.


In [ ]:
decisions = result.decisions.assign(
    gained=gained,
    lost=lost,
    routed=routed,
)
dataset_outcomes = decisions.groupby("dataset").agg(
    prompts=("dataset", "size"),
    routed=("routed", "sum"),
    gains=("gained", "sum"),
    losses=("lost", "sum"),
    truncated=("router_was_truncated", "sum"),
)
dataset_outcomes["net_answers"] = (
    dataset_outcomes.gains - dataset_outcomes.losses
)
dataset_outcomes["routed_fraction"] = (
    dataset_outcomes.routed / dataset_outcomes.prompts
)
router_dataset_metrics = result.per_dataset_metrics.loc[
    result.per_dataset_metrics.strategy.eq("modernbert_qwen_tier_router")
].set_index("dataset")
dataset_outcomes = dataset_outcomes.join(
    router_dataset_metrics[
        ["resource_savings", "conservative_resource_savings"]
    ]
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
net_order = dataset_outcomes.sort_values("net_answers")
axes[0].barh(
    net_order.index,
    net_order.net_answers,
    color=np.where(net_order.net_answers >= 0, "tab:green", "tab:red"),
)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set(title="Net correct answers by dataset", xlabel="Gains minus losses")
savings_order = dataset_outcomes.sort_values("conservative_resource_savings")
axes[1].barh(
    savings_order.index,
    100 * savings_order.conservative_resource_savings,
    color=np.where(
        savings_order.conservative_resource_savings >= 0,
        "tab:blue",
        "tab:orange",
    ),
)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set(title="Savings at 20 ms overhead", xlabel="Analytical savings (%)")
plt.tight_layout()
plt.show()

display(dataset_outcomes.sort_values("net_answers"))
display(
    decisions.groupby("selected_model").agg(
        prompts=("selected_model", "size"),
        gains=("gained", "sum"),
        losses=("lost", "sum"),
    )
)
display(
    decisions.groupby("router_was_truncated").agg(
        prompts=("dataset", "size"),
        routed=("routed", "sum"),
        gains=("gained", "sum"),
        losses=("lost", "sum"),
    )
)


## 13. Export the reconstructable artifact and optional demo

The ZIP contains the scored candidate evidence, pinned evidence contract,
setup comparison, threshold frontiers, sealed-test decisions, ModernBERT
adapter and heads, Platt parameters, analytical scenario, and timing
diagnostics. The Gradio demo loads no Qwen weights; it shows the routing
decision and analytical candidate latency.


In [ ]:
report_dir = export_public_benchmark(result, scenario, OUTPUT_DIR)
sensitivity.to_csv(report_dir / "validation_sensitivity.csv", index=False)
setup_comparison.to_csv(report_dir / "setup_comparison.csv", index=False)
setup_threshold_search.to_csv(
    report_dir / "setup_threshold_search.csv", index=False
)
records.to_parquet(report_dir / "qwen_candidate_records.parquet", index=False)
panel_summary.to_csv(report_dir / "qwen_candidate_panel_summary.csv")
(report_dir / "qwen_evidence_contract.json").write_text(
    json.dumps(
        {**evidence_contract, "evidence_tag": EVIDENCE_TAG},
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)
for setup_name, training in trainings.items():
    setup_dir = report_dir / "setup_diagnostics" / setup_name
    setup_dir.mkdir(parents=True, exist_ok=True)
    training.history.to_csv(setup_dir / "training_history.csv", index=False)
    training.calibration_diagnostics.to_csv(
        setup_dir / "calibration_diagnostics.csv", index=False
    )

artifact_dir = export_modernbert_hybrid_poc(
    selected_training,
    panel.models,
    report_dir / "modernbert_router",
    selected_threshold=result.selected_threshold,
    router_active=result.router_active,
    poc_passed=result.single_run_passed,
    failure_reasons=result.failure_reasons,
    minimum_predicted_savings=DEFAULT_CONFIG.minimum_predicted_speedup,
    validation_quality_margin=DEFAULT_CONFIG.validation_quality_margin,
    minimum_macro_quality_retention=DEFAULT_CONFIG.minimum_macro_quality_retention,
    maximum_quality_loss_rate_ucl=DEFAULT_CONFIG.maximum_quality_loss_rate_ucl,
    minimum_routed_safety_precision_lcb=(
        DEFAULT_CONFIG.minimum_routed_safety_precision_lcb
    ),
    minimum_guarded_dataset_quality_retention_lcb=(
        DEFAULT_CONFIG.minimum_guarded_dataset_quality_retention_lcb
    ),
    minimum_guarded_dataset_prompts=DEFAULT_CONFIG.minimum_guarded_dataset_prompts,
    conservative_router_overhead_s=DEFAULT_CONFIG.conservative_router_overhead_s,
    minimum_consecutive_feasible_thresholds=(
        DEFAULT_CONFIG.minimum_consecutive_feasible_thresholds
    ),
    benchmark_fingerprint=result.benchmark_fingerprint,
    setup_name=SELECTED_SETUP,
    oracle_auxiliary_weight=selected_spec["oracle_auxiliary_weight"],
    router_overhead_benchmark=(
        router_overhead_benchmark.summary
        if router_overhead_benchmark is not None
        else None
    ),
    config=selected_config,
)
overhead_comparison.to_csv(
    report_dir / "modernbert_overhead_comparison.csv", index=False
)
if router_overhead_benchmark is not None:
    router_overhead_benchmark.export(report_dir)

demo_runtime = HybridModernBERTRouterRuntime.from_training_result(
    selected_training,
    model_names=panel.models,
    fallback_model=result.fallback_model,
    selected_threshold=result.selected_threshold,
    router_active=result.router_active,
    minimum_predicted_savings=DEFAULT_CONFIG.minimum_predicted_speedup,
    scenario=scenario,
    config=selected_config,
    device=DEVICE,
)
demo = create_gradio_demo(demo_runtime)
if LAUNCH_INTERACTIVE_DEMO:
    demo.launch(share=True, debug=False, prevent_thread_lock=True)

bundle_path = shutil.make_archive(str(OUTPUT_DIR.resolve()), "zip", root_dir=OUTPUT_DIR)
print("Reports:", report_dir)
print("Router artifact:", artifact_dir)
print("ZIP:", bundle_path)
try:
    from google.colab import files

    files.download(bundle_path)
except ImportError:
    pass


## 14. Final interpretation checklist

Before a LinkedIn or investor claim, confirm all of the following:

- the evidence tag and pinned model/dataset revisions are in the ZIP;
- all 2,700 candidate outcomes exist and malformed-answer rates are visible;
- setup and threshold selection used validation only;
- at least two neighboring thresholds passed, or the router failed closed;
- the sealed test opened once and `single_run_passed` is explicit;
- quality-retention, macro, harm, precision, subgroup, and savings gates pass;
- ModernBERT p50 and p95 are compared with break-even overhead;
- candidate latency is described as analytical, not measured production latency;
- no one dataset supplies all net gains and no tier has a zero route rate;
- seeds 42/43/44 are reported together without cherry-picking; and
- dataset-OOD evidence is reported separately.

A useful numerical investor summary has both sides: “the router sent X% of
prompts to smaller tiers and saved Y% analytical latency, while its 95%
quality-retention lower bound was Z%.” Never present Y without Z, and never
turn analytical milliseconds into guaranteed dollars without a customer
hardware calibration.
